# Setup Cells

In [2]:
#for Pandas like formatting
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true") #enabling arrow for speed
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", "50")
# spark.conf.set("spark.sql.repl.eagerEval.truncate", "0")

In [4]:
spark.sql("USE prod_db")  # your run_ddl.py schema is prod_db
# spark.sql("SELECT * FROM prod_db.customer LIMIT 10").show()

""


In [6]:
%%sql 
show catalogs

,catalog
0,local
1,spark_catalog


In [7]:
%%sql
show schemas in local;

,namespace
0,prod_db


In [8]:
%%sql
show schemas in spark_catalog;

,namespace
0,default


In [9]:
%%sql
show tables in prod_db;

,namespace,tableName,isTemporary
0,prod_db,orders,False
1,prod_db,partsupp,False
2,prod_db,supplier,False
3,prod_db,nation,False
4,prod_db,lineitem,False
5,prod_db,customer,False
6,prod_db,part,False
7,prod_db,region,False


## 1.11 Exercises

In [82]:
%%sql
-- Question 1
select r_name,
--count(l.l_orderkey)
count(l_orderkey||l_partkey) as item_value 
from lineitem l
left outer join orders o on l.l_orderkey = o.o_orderkey
left outer join customer c on o_custkey = c_custkey
left outer join nation n  on n_nationkey = c_nationkey
left outer join region r on n_regionkey = r_regionkey
where l_returnflag in ("R" )
group by 1
order by 2 desc
;

,r_name,item_value
0,MIDDLE EAST,30414
1,ASIA,29786
2,AFRICA,29624
3,EUROPE,29259
4,AMERICA,29218


In [69]:
%%sql
-- Question 2
-- List the top 10 most selling parts (part name)

select p.p_name
from orders o 
left join lineitem l on l.l_orderkey = o.o_orderkey
left join part p on p.p_partkey = l.l_partkey
group by 1
order by count(l_partkey) desc
limit 10

,p_name
0,floral azure papaya moccasin indian
1,almond medium midnight cornflower beige
2,hot blue honeydew salmon slate
3,maroon midnight burnished indian blanched
4,misty burlywood wheat lavender beige
5,chiffon purple sky bisque dark
6,pink lime grey gainsboro deep
7,rose wheat chiffon aquamarine yellow
8,bisque spring dim khaki pale
9,hot floral blanched lime rose


In [73]:
%%sql
-- Question 3
-- Sellers (name) who have sold at least one of the top 10 selling parts

with setup as (
    select p.p_name,
    p.p_partkey
    from orders o 
    left join lineitem l on l.l_orderkey = o.o_orderkey
    left join part p on p.p_partkey = l.l_partkey
    group by 1,2
    order by count(l_partkey) desc
    limit 10
)
select distinct s.s_name
from setup sp
left join partsupp ps on sp.p_partkey = ps.ps_partkey
left join supplier s on s.s_suppkey = ps.ps_suppkey
;

,s_name
0,Supplier#000000401
1,Supplier#000000141
2,Supplier#000000881
3,Supplier#000000621
4,Supplier#000000909
5,Supplier#000000653
6,Supplier#000000397
7,Supplier#000000826
8,Supplier#000000568
9,Supplier#000000310


In [77]:
%%sql
-- Question 4
-- Number of items returned for each order price bucket. The definition of order price bucket is shown below.

select 
CASE
    WHEN o_totalprice > 100000 THEN 'high'
    WHEN o_totalprice BETWEEN 25000 AND 100000 THEN 'medium'
    ELSE 'low'
END AS order_price_bucket,
count(l.l_partkey) as item_count_returned
from orders o 
left join lineitem l on o.o_orderkey = l.l_orderkey
where l.l_returnflag in ("R")
group by 1
order by 2 desc
;

,order_price_bucket,item_count_returned
0,high,121779
1,medium,23777
2,low,2745


In [84]:
%%sql
-- Question 5
-- Number of items returned for each order price bucket. The definition of order price bucket is shown below.

with transit_dates as (
    select 
    *,
    datediff(l.l_receiptdate,l.l_shipdate) as l_transitdays
    from lineitem l
)
    select n.n_name,
    round(avg(l_transitdays*1.00),2) as avg_transit_days
    from transit_dates td 
    left join orders o on td.l_orderkey = o.o_orderkey
    left join customer c on o.o_custkey = c.c_custkey
    left join nation n on c.c_nationkey = n.n_nationkey
    group by 1
    order by 2 desc
;

,n_name,avg_transit_days
0,KENYA,15.67
1,CHINA,15.60
2,MOZAMBIQUE,15.59
3,FRANCE,15.58
4,UNITED KINGDOM,15.53
5,SAUDI ARABIA,15.53
6,IRAQ,15.53
7,GERMANY,15.53
8,INDIA,15.52
9,PERU,15.52


## 2.4 Exercises

In [86]:
%%sql
-- Question 1
-- Sellers who have sold at least one of the top 10 selling parts (using CTE)
with top_ten_selling_parts as (
    select l.l_partkey,
    count(*) as parts_ct
    from lineitem l 
    group by 1
    order by 2 desc
    limit 10
)
    select distinct
    s.s_name
    from top_ten_selling_parts tts
    left join lineitem l on l.l_partkey = tts.l_partkey
    left join supplier s on s.s_suppkey = l.l_suppkey
;

,s_name
0,Supplier#000000621
1,Supplier#000000141
2,Supplier#000000881
3,Supplier#000000401
4,Supplier#000000653
5,Supplier#000000397
6,Supplier#000000909
7,Supplier#000000115
8,Supplier#000000850
9,Supplier#000000380


## 3.6 Exercises

Window functions have four parts:
* Partition - specifying column values that define what rows are affected
* Order By - optional clause to order the rows within the partition
* Function - function applied to the current row
* Window Frame - specifying which rows are considered for the function


In window frames rows between specifies the specific rows relative to the current row while range between specifies the values relative to that of the current row.

In [91]:
%%sql
-- Let's look at an example showing the difference between RANK, DENSE_RANK and ROW_NUMBER
SELECT 
    order_date,
    order_id,
    total_price,
    ROW_NUMBER() OVER (PARTITION BY order_date ORDER BY total_price) AS row_number,
    RANK() OVER (PARTITION BY order_date ORDER BY total_price) AS rank,
    DENSE_RANK() OVER (PARTITION BY order_date ORDER BY total_price) AS dense_rank,
    PERCENT_RANK() OVER (PARTITION BY order_date ORDER BY total_price) AS percent_rank,
    NTILE(5) OVER (PARTITION BY order_date ORDER BY total_price) AS ntile,
    CUME_DIST() OVER (PARTITION BY order_date ORDER BY total_price) AS cume_dist

FROM (
    SELECT 
        '2024-07-08' AS order_date, 'order_1' AS order_id, 100 AS total_price UNION ALL
    SELECT 
        '2024-07-08', 'order_2', 200 UNION ALL
    SELECT 
        '2024-07-08', 'order_3', 150 UNION ALL
    SELECT 
        '2024-07-08', 'order_4', 90 UNION ALL
    SELECT 
        '2024-07-08', 'order_5', 100 UNION ALL
    SELECT 
        '2024-07-08', 'order_6', 90 UNION ALL
    SELECT 
        '2024-07-08', 'order_7', 100 UNION ALL
    SELECT 
        '2024-07-10', 'order_8', 100 UNION ALL
    SELECT 
        '2024-07-10', 'order_9', 100 UNION ALL
    SELECT 
        '2024-07-10', 'order_10', 100 UNION ALL
    SELECT 
        '2024-07-11', 'order_11', 100
) AS orders
ORDER BY order_date, row_number;

,order_date,order_id,total_price,row_number,rank,dense_rank,percent_rank,ntile,cume_dist
0,2024-07-08,order_4,90,1,1,1,0.000000,1,0.285714
1,2024-07-08,order_6,90,2,1,1,0.000000,1,0.285714
2,2024-07-08,order_1,100,3,3,2,0.333333,2,0.714286
3,2024-07-08,order_5,100,4,3,2,0.333333,2,0.714286
4,2024-07-08,order_7,100,5,3,2,0.333333,3,0.714286
5,2024-07-08,order_3,150,6,6,3,0.833333,4,0.857143
6,2024-07-08,order_2,200,7,7,4,1.000000,5,1.000000
7,2024-07-10,order_8,100,1,1,1,0.000000,1,1.000000
8,2024-07-10,order_9,100,2,1,1,0.000000,2,1.000000
9,2024-07-10,order_10,100,3,1,1,0.000000,3,1.000000


In [112]:
%%sql
-- Question 1
-- Write a query on the orders table that has the following output:
select
    cast(date_trunc('month', o_orderdate) as date) as order_month,
    o_custkey,
    o_totalprice,
    avg(o_totalprice) over (
        partition by o_custkey 
        order by cast(date_trunc('month', o_orderdate) as date)
        rows between 1 preceding and 1 following
    )as three_mo_total_price_avg,
    avg(o_totalprice) over (
        partition by o_custkey 
        order by cast(date_trunc('month', o_orderdate) as date)
        range between interval '1' MONTH preceding and interval '1' MONTH following
    )as consecutive_three_mo_total_price_avg
from orders o
group by 1,2,3
;

,order_month,o_custkey,o_totalprice,three_mo_total_price_avg,consecutive_three_mo_total_price_avg
0,1992-04-01,1,283261.47,183520.365000,183520.365000
1,1992-04-01,1,83779.26,210096.090000,183520.365000
2,1993-06-01,1,263247.54,127516.616667,263247.540000
3,1994-12-01,1,35523.05,177975.270000,35523.050000
4,1995-11-01,1,235155.22,117388.036667,235155.220000
...,...,...,...,...,...
995,1994-11-01,227,233327.80,179162.366667,233327.800000
996,1995-02-01,227,112666.30,206180.710000,192607.165000
997,1995-03-01,227,272548.03,195540.526667,192607.165000
998,1996-04-01,227,201407.25,188053.106667,201407.250000


In [120]:
%%sql
-- Question 2
-- From the orders table get the 3 lowest spending customers per day

with customer_daily_spend as (
    select
        o_orderdate,
        o_custkey,
        sum(o_totalprice) as daily_spend
    from orders o
    group by 1,2
),
customer_daily_ranking as (
    select
    *,
    row_number () over (partition by o_orderdate order by daily_spend) as ranking
    from customer_daily_spend
)
select * from customer_daily_ranking 
where ranking <4
order by o_orderdate
;

,o_orderdate,o_custkey,daily_spend,ranking
0,1992-01-01,9376,12781.38,1
1,1992-01-01,9103,25496.77,2
2,1992-01-01,14218,30944.67,3
3,1992-01-02,2746,16908.46,1
4,1992-01-02,4355,25445.96,2
...,...,...,...,...
995,1992-11-27,11354,15368.91,3
996,1992-11-28,11575,1375.40,1
997,1992-11-28,3830,6829.02,2
998,1992-11-28,9565,18583.38,3


In [124]:
%%sql
-- Question 3
-- Write a SQL query using the orders table that calculates the following columns:\

with customer_daily_spend as (
    select
        o_orderdate,
        o_custkey,
        sum(o_totalprice) as o_today_total_price
    from orders o
    group by 1,2
)
    select
    o_orderdate,
    o_custkey,
    o_today_total_price as o_totalprice,
    o_today_total_price - lag(o_today_total_price) over( partition by o_custkey order by o_orderdate) as totalprice_diff
    from customer_daily_spend
    order by 2,1
;

,o_orderdate,o_custkey,o_totalprice,totalprice_diff
0,1992-04-19,1,83779.26,None
1,1992-04-26,1,283261.47,199482.21
2,1993-06-22,1,263247.54,-20013.93
3,1994-12-24,1,35523.05,-227724.49
4,1995-11-01,1,235155.22,199632.17
...,...,...,...,...
995,1995-01-20,104,154485.27,85406.62
996,1995-06-05,104,95192.41,-59292.86
997,1995-11-30,104,204434.71,109242.30
998,1998-02-06,104,86606.20,-117828.51
